In [ ]:
import pandas as pd
from utils import json_to_df, read_jsons, inject_prompt_type

In [7]:
BASE_PATH = "../../results/sbrc/cifar10/exp2-models/{prompt_type}/{model_name}/{selection_algorithm}"

In [8]:
MODELS_NAME = ["qwen3:8b", "llama3.2:3b", "llama3.1:8b"]
PROMPTS_TYPE = ['cot', 'description', 'fewshot']
SELECTION_ALGORTIHM = ['poc', 'oort', 'random', 'rrobin']

In [9]:
dfs = []
for mn in MODELS_NAME:
    for sa in SELECTION_ALGORTIHM:
        for pt in PROMPTS_TYPE:
            js_list = read_jsons(BASE_PATH.format(prompt_type=pt, model_name=mn, selection_algorithm=sa))
            if len(js_list)>0:
                df = pd.concat([inject_prompt_type(json_to_df(js), pt) for js in js_list])
                dfs.append(df)

In [11]:
data = pd.concat(dfs)

In [12]:
data['prompt_type'].unique()

array(['chain-of-thought', 'description-only', 'few-shot'], dtype=object)

In [13]:
data['hue_column'] = data['prompt_type'] +'-'+ data['selection_algorithm'] +'-'+ data['llm_model_name']
data = data.loc[data['round'] > 1].sort_values('hue_column')

In [14]:
data = data.loc[data['round'] > 1].sort_values('hue_column')
counts = data.groupby(['hue_column', 'round']).size()

In [15]:
data = data.loc[data['round'] > 1].sort_values('hue_column')
avg_accuracy = data.groupby(by=['hue_column', 'round'])['eval_accuracy'].mean().reset_index()
std_accuracy = data.groupby(by=['hue_column', 'round'])['eval_accuracy'].std().reset_index()
idxmax = avg_accuracy.groupby(['hue_column'])['eval_accuracy'].idxmax()

In [16]:
# std_accuracy.iloc[idxmax]

In [17]:
avg_sample_time = data.groupby(by=['hue_column', 'round'])['sample_time'].mean().reset_index()

In [18]:
total_params = 62006
size_mb = 0.24

In [19]:
selected = data.loc[data['selected']]
amount_selected = (selected.groupby(['round', 'hue_column']).size()/3).reset_index()
amount_selected = amount_selected.loc[amount_selected['round'] > 1]
# amount_selected.groupby(['hue_column']).mean().reset_index()

In [20]:
selected = data.loc[data['selected']]
amount_selected = (selected.groupby(['round', 'hue_column']).size()/3).reset_index()
amount_selected = amount_selected.loc[amount_selected['round'] > 1]
amount_selected['download_mb'] = amount_selected[0]*size_mb
# amount_selected.groupby(['hue_column']).sum().reset_index()[['hue_column', 'download_mb']]

In [21]:
d = pd.DataFrame({
    "sel":amount_selected.groupby(['hue_column']).sum().reset_index()['hue_column'],
    "accuracy": avg_accuracy.groupby(['hue_column'])['eval_accuracy'].max().reset_index()['eval_accuracy'],
    "std_accuracy": std_accuracy.iloc[idxmax].reset_index()['eval_accuracy'],
    "k_medio": amount_selected.groupby(['hue_column']).mean().reset_index()[0],
    "k_std": amount_selected.groupby(['hue_column']).std().reset_index()[0],
    "download_mb": amount_selected.groupby(['hue_column']).sum().reset_index()['download_mb'],
    "sample_time": avg_sample_time.groupby(['hue_column'])['sample_time'].mean().reset_index()['sample_time'],
})

In [22]:
mask = d['sel'].apply(lambda x: 'poc' in x)
d.loc[mask]

,sel,accuracy,std_accuracy,k_medio,k_std,download_mb,sample_time
3,chain-of-thought-poc-llama3.1:8b,0.381333,0.124904,8.469388,2.328287,99.60,0.530703
4,chain-of-thought-poc-llama3.2:3b,0.354400,0.158398,9.387755,1.656003,110.40,0.201960
5,chain-of-thought-poc-qwen3:8b,0.392267,0.138111,7.605442,5.285670,89.44,1.798559
15,description-only-poc-llama3.1:8b,0.373600,0.139406,10.000000,0.000000,117.60,0.180500
16,description-only-poc-llama3.2:3b,0.360133,0.138627,10.000000,0.000000,117.60,0.290017
17,description-only-poc-qwen3:8b,0.385467,0.137067,7.687075,5.666833,90.40,3.799290
24,few-shot-poc-llama3.1:8b,0.376933,0.134053,8.741497,6.179890,102.80,3.212504
25,few-shot-poc-llama3.2:3b,0.351733,0.135001,5.000000,0.000000,58.80,0.310371
26,few-shot-poc-qwen3:8b,0.384667,0.134540,6.442177,4.732369,75.76,1.434668


In [24]:
d.to_csv('./data/k_agent.csv', index=False)